In [5]:
# -----------------------------------------------------------
# CELL 1: SETUP & VISUALIZATION TOOLS (PROFESSIONAL STYLE)
# -----------------------------------------------------------
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import pickle
import json
from datetime import datetime
from google.colab import drive

# 1. Mount Drive
if os.path.ismount('/content/drive'):
    print("✓ Drive already mounted.")
else:
    drive.mount('/content/drive', force_remount=True)

# 2. Paths
BASE_DIR = "/content/drive/MyDrive/FabricDefect"
FEATURE_DIR = f"{BASE_DIR}/features"
MODEL_DIR = f"{BASE_DIR}/models"
BACKUP_DIR = f"{BASE_DIR}/backups"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

# 3. Advanced Plotting Configuration
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

# 4. Custom Progress Bar
def print_progress(iteration, total, prefix='', suffix='', length=40):
    percent = ("{0:.1f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = '█' * filled_length + '-' * (length - filled_length)
    print(f'\r{prefix} |{bar}| {percent}% {suffix}', end='\r')
    if iteration == total:
        print()

# 5. Modern Confusion Matrix Plotter
def plot_advanced_confusion_matrix(cm, classes, title='Confusion Matrix', color_theme='Blues'):
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest', cmap=getattr(plt.cm, color_theme))

    # Add colorbar
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Count", rotation=-90, va="bottom")

    # Add ticks
    ax.set_xticks(np.arange(cm.shape[1]))
    ax.set_yticks(np.arange(cm.shape[0]))
    ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(classes, fontsize=10)

    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')

    # Add text annotations with contrast color logic
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm[i, j] > thresh else "black"
            ax.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color=text_color,
                    fontsize=11, fontweight='bold')

    # Add grid for readability
    ax.set_xticks(np.arange(cm.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(cm.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.2)
    ax.tick_params(which="minor", bottom=False, left=False)

    plt.tight_layout()
    plt.show()

print("✓  Visualization & Setup Complete.")

✓ Drive already mounted.
✓  Visualization & Setup Complete.


In [6]:
# -----------------------------------------------------------
# CELL 2: LOAD DATA & MATH KERNEL
# -----------------------------------------------------------
print("Loading processed data...")
X_train = np.load(f"{FEATURE_DIR}/final_X_train.npy")
y_train = np.load(f"{FEATURE_DIR}/final_y_train.npy")
X_val   = np.load(f"{FEATURE_DIR}/final_X_val.npy")
y_val   = np.load(f"{FEATURE_DIR}/final_y_val.npy")
X_test  = np.load(f"{FEATURE_DIR}/final_X_test.npy")
y_test  = np.load(f"{FEATURE_DIR}/final_y_test.npy")

# Load Metadata
with open(f"{FEATURE_DIR}/dataset_metadata.pkl", "rb") as f:
    meta = pickle.load(f)
class_names = list(meta['class_to_idx'].keys())
NUM_CLASSES = len(class_names)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Classes ({NUM_CLASSES}): {class_names}")

# --- MATH HELPERS (FROM SCRATCH) ---

def sigmoid(z):
    # Numerically stable sigmoid
    return np.where(z >= 0,
                    1 / (1 + np.exp(-z)),
                    np.exp(z) / (1 + np.exp(z)))

def softmax(z):
    # Numerically stable softmax
    z_max = np.max(z, axis=1, keepdims=True)
    e_z = np.exp(z - z_max)
    return e_z / np.sum(e_z, axis=1, keepdims=True)

def to_one_hot(y, n_classes):
    n_samples = y.shape[0]
    one_hot = np.zeros((n_samples, n_classes))
    one_hot[np.arange(n_samples), y] = 1
    return one_hot

def get_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

print("✓ Math Kernel Ready.")

Loading processed data...
Train: (1756, 2048) | Val: (435, 2048) | Test: (548, 2048)
Classes (6): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
✓ Math Kernel Ready.


In [7]:
# -----------------------------------------------------------
# CELL 3: CORE TREE STRUCTURE & BASE LEARNER
# -----------------------------------------------------------

class TreeNode:
    def __init__(self, depth=0):
        self.depth = depth
        self.left = None
        self.right = None
        self.feature_idx = None
        self.threshold = None
        self.value = None  # Leaf weight
        self.is_leaf = False

class RegularizedTree:
    """
    A Regression Tree that learns from Gradients (G) and Hessians (H).
    Implements XGBoost-style Gain calculation with Lambda (L2 reg) and Gamma (min gain).
    """
    def __init__(self, max_depth=3, lambda_reg=1.0, gamma=0.0, min_child_weight=1.0, feature_subsample=1.0):
        self.max_depth = max_depth
        self.lambda_reg = lambda_reg
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.feature_subsample = feature_subsample
        self.root = None

    def _calc_leaf_weight(self, grad, hess):
        # XGBoost formula: w = - sum(G) / (sum(H) + lambda)
        return -np.sum(grad) / (np.sum(hess) + self.lambda_reg)

    def _calc_structure_score(self, grad, hess):
        # Score = (sum(G)^2) / (sum(H) + lambda)
        return (np.sum(grad)**2) / (np.sum(hess) + self.lambda_reg)

    def _find_best_split(self, X, grad, hess):
        n_samples, n_features = X.shape

        if n_samples <= 1: return None, None

        # Calculate score of current node (before split)
        current_score = self._calc_structure_score(grad, hess)

        best_gain = 0
        best_feat = None
        best_thresh = None

        # Feature Subsampling (Column sampling)
        feat_idxs = np.random.choice(n_features, int(n_features * self.feature_subsample), replace=False)

        # Calculate sum of gradients/hessians for the whole node once
        G_node = np.sum(grad)
        H_node = np.sum(hess)

        for feat_idx in feat_idxs:
            # Unique values for potential thresholds
            # Histogram approximation for speed: limit checkpoints
            unique_vals = np.unique(np.percentile(X[:, feat_idx], np.linspace(0, 100, 20)))

            for thresh in unique_vals:
                # Mask for left child
                left_mask = X[:, feat_idx] <= thresh

                # Skip invalid splits (empty children)
                if not np.any(left_mask) or np.all(left_mask):
                    continue

                # Calculate sums for children
                G_L = np.sum(grad[left_mask])
                H_L = np.sum(hess[left_mask])

                # Pruning: Minimum Child Weight check (Hessian sum)
                if H_L < self.min_child_weight: continue

                G_R = G_node - G_L
                H_R = H_node - H_L

                if H_R < self.min_child_weight: continue

                # Calculate Gain
                # Gain = 0.5 * [ Score_L + Score_R - Score_Node ] - Gamma
                score_L = (G_L**2) / (H_L + self.lambda_reg)
                score_R = (G_R**2) / (H_R + self.lambda_reg)

                gain = 0.5 * (score_L + score_R - current_score) - self.gamma

                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat_idx
                    best_thresh = thresh

        if best_gain > 0:
            return best_feat, best_thresh
        return None, None

    def fit(self, X, grad, hess):
        self.root = self._build_tree(X, grad, hess, depth=0)

    def _build_tree(self, X, grad, hess, depth):
        node = TreeNode(depth)

        # Check stopping criteria
        if depth >= self.max_depth or len(X) < 2:
            node.is_leaf = True
            node.value = self._calc_leaf_weight(grad, hess)
            return node

        # Find best split
        best_feat, best_thresh = self._find_best_split(X, grad, hess)

        if best_feat is None:
            node.is_leaf = True
            node.value = self._calc_leaf_weight(grad, hess)
        else:
            node.feature_idx = best_feat
            node.threshold = best_thresh

            left_mask = X[:, best_feat] <= best_thresh
            right_mask = ~left_mask

            node.left = self._build_tree(X[left_mask], grad[left_mask], hess[left_mask], depth + 1)
            node.right = self._build_tree(X[right_mask], grad[right_mask], hess[right_mask], depth + 1)

        return node

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.is_leaf:
            return node.value

        if x[node.feature_idx] <= node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

print("✓ Regularized Regression Tree Defined.")

✓ Regularized Regression Tree Defined.


In [8]:
# -----------------------------------------------------------
# CELL 4: CUSTOM XGBOOST-STYLE BOOSTER
# -----------------------------------------------------------
class CustomXGBoost:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, reg_lambda=1.0, gamma=0.0):
        self.K = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.trees = [] # List of lists: [ [Tree_Class1, Tree_Class2...], [Iter2...] ]
        self.history = {'loss': [], 'acc': []}

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples = X.shape[0]
        self.n_classes = len(np.unique(y))
        y_hot = to_one_hot(y, self.n_classes)

        # Initial Prediction (F_0 = 0) -> Softmax -> Uniform Prob
        self.scores = np.zeros((n_samples, self.n_classes))

        print(f"Training Custom XGBoost ({self.K} rounds)...")
        start_time = time.time()

        for i in range(self.K):
            probs = softmax(self.scores)

            # Multiclass Log-Loss Gradients & Hessians
            # Grad = p - y
            # Hess = p * (1 - p)  (Diagonal approximation for speed)
            grads = probs - y_hot
            hesses = probs * (1 - probs)

            # Prevent hessian from being too small (division by zero risk)
            hesses = np.maximum(hesses, 1e-4)

            current_iter_trees = []

            # Build one tree per class (One-vs-Rest Boosting)
            for k in range(self.n_classes):
                tree = RegularizedTree(
                    max_depth=self.max_depth,
                    lambda_reg=self.reg_lambda,
                    gamma=self.gamma,
                    feature_subsample=0.8
                )

                tree.fit(X, grads[:, k], hesses[:, k])

                # Update scores with shrinkage
                update = tree.predict(X)
                self.scores[:, k] += self.lr * update

                current_iter_trees.append(tree)

            self.trees.append(current_iter_trees)

            # Evaluation
            train_acc = np.mean(np.argmax(self.scores, axis=1) == y)

            msg = f"Acc: {train_acc:.2%}"
            if X_val is not None:
                val_pred = self.predict(X_val)
                val_acc = np.mean(val_pred == y_val)
                self.history['acc'].append(val_acc)
                msg += f" | Val: {val_acc:.2%}"

            print_progress(i+1, self.K, prefix="[XGB]", suffix=msg)

        self.train_time = time.time() - start_time

    def predict_proba(self, X):
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [9]:
# -----------------------------------------------------------
# CELL 5: CUSTOM LIGHTGBM-STYLE BOOSTER (GOSS)
# -----------------------------------------------------------
class CustomLightGBM:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3, goss_alpha=0.2, goss_beta=0.2):
        self.K = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.goss_alpha = goss_alpha # Retain top alpha% gradients
        self.goss_beta = goss_beta   # Sample beta% of small gradients
        self.trees = []
        self.history = {'acc': []}

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples = X.shape[0]
        self.n_classes = len(np.unique(y))
        y_hot = to_one_hot(y, self.n_classes)
        self.scores = np.zeros((n_samples, self.n_classes))

        print(f"Training Custom LightGBM (GOSS)...")
        start_time = time.time()

        for i in range(self.K):
            probs = softmax(self.scores)
            grads = probs - y_hot
            hesses = probs * (1 - probs)
            hesses = np.maximum(hesses, 1e-4)

            current_iter_trees = []

            for k in range(self.n_classes):
                g_k = grads[:, k]
                h_k = hesses[:, k]

                # --- GOSS LOGIC ---
                abs_grad = np.abs(g_k)
                n_large = int(n_samples * self.goss_alpha)
                n_small = int(n_samples * self.goss_beta)

                # Sort gradients
                sorted_indices = np.argsort(-abs_grad)

                top_set = sorted_indices[:n_large]
                # Random sample from the rest
                rand_set = np.random.choice(sorted_indices[n_large:], n_small, replace=False)

                used_indices = np.concatenate([top_set, rand_set])

                # Weight amplification for small gradients: (1-a)/b
                amp_factor = (1 - self.goss_alpha) / self.goss_beta

                # Construct GOSS dataset for this tree
                X_goss = X[used_indices]
                g_goss = g_k[used_indices].copy()
                h_goss = h_k[used_indices].copy()

                # Apply amplification to the random set (the second part of the array)
                g_goss[n_large:] *= amp_factor
                h_goss[n_large:] *= amp_factor

                # Fit tree on smaller GOSS subset
                tree = RegularizedTree(max_depth=self.max_depth, lambda_reg=0.1)
                tree.fit(X_goss, g_goss, h_goss)

                # Update scores for ALL instances
                update = tree.predict(X)
                self.scores[:, k] += self.lr * update

                current_iter_trees.append(tree)

            self.trees.append(current_iter_trees)

            # Evaluation
            train_acc = np.mean(np.argmax(self.scores, axis=1) == y)
            msg = f"Acc: {train_acc:.2%}"
            if X_val is not None:
                val_acc = np.mean(self.predict(X_val) == y_val)
                self.history['acc'].append(val_acc)
                msg += f" | Val: {val_acc:.2%}"

            print_progress(i+1, self.K, prefix="[LGBM]", suffix=msg)

        self.train_time = time.time() - start_time

    def predict_proba(self, X):
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [10]:
# -----------------------------------------------------------
# CELL 6: CUSTOM CATBOOST-STYLE BOOSTER (ORDERED)
# -----------------------------------------------------------
class CustomCatBoost:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.K = n_estimators
        self.lr = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.history = {'acc': []}

    def fit(self, X, y, X_val=None, y_val=None):
        n_samples = X.shape[0]
        self.n_classes = len(np.unique(y))
        y_hot = to_one_hot(y, self.n_classes)
        self.scores = np.zeros((n_samples, self.n_classes))

        # Generate a fixed random permutation for Ordered Boosting
        perm = np.random.permutation(n_samples)
        X_perm = X[perm]
        y_hot_perm = y_hot[perm]

        print(f"Training Custom CatBoost (Ordered)...")
        start_time = time.time()

        for i in range(self.K):
            # Update probabilities
            probs = softmax(self.scores)

            # --- ORDERED BOOSTING SIMULATION ---
            # In true CatBoost, separate models are maintained.
            # Here, we approximate by training on a growing prefix of data
            # to ensure the model doesn't see "future" gradients in the permutation chain.

            # Calculate dynamic subset size (growing as training progresses)
            # Starts at 50% data, grows to 100%
            subset_ratio = 0.5 + 0.5 * (i / self.K)
            n_subset = int(n_samples * subset_ratio)

            # Use permuted data subset
            curr_indices = perm[:n_subset]
            X_curr = X[curr_indices]

            # Calculate gradients on the FULL dataset first based on current state
            # But we will only FIT the tree on the subset
            grads_full = probs - y_hot
            hesses_full = probs * (1 - probs)
            hesses_full = np.maximum(hesses_full, 1e-4)

            current_iter_trees = []

            for k in range(self.n_classes):
                # Get gradients for the subset
                g_sub = grads_full[curr_indices, k]
                h_sub = hesses_full[curr_indices, k]

                # Fit tree on ordered subset
                tree = RegularizedTree(max_depth=self.max_depth, lambda_reg=0.5)
                tree.fit(X_curr, g_sub, h_sub)

                # Update scores for ALL data
                update = tree.predict(X)
                self.scores[:, k] += self.lr * update

                current_iter_trees.append(tree)

            self.trees.append(current_iter_trees)

            # Evaluation
            train_acc = np.mean(np.argmax(self.scores, axis=1) == y)
            msg = f"Acc: {train_acc:.2%}"
            if X_val is not None:
                val_acc = np.mean(self.predict(X_val) == y_val)
                self.history['acc'].append(val_acc)
                msg += f" | Val: {val_acc:.2%}"

            print_progress(i+1, self.K, prefix="[CatB]", suffix=msg)

        self.train_time = time.time() - start_time

    def predict_proba(self, X):
        scores = np.zeros((X.shape[0], self.n_classes))
        for iter_trees in self.trees:
            for k, tree in enumerate(iter_trees):
                scores[:, k] += self.lr * tree.predict(X)
        return softmax(scores)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [11]:
# -----------------------------------------------------------
# CELL 6.5: MISSING EVALUATION HELPERS (Run this before Cell 7)
# -----------------------------------------------------------
import numpy as np

def get_confusion_matrix(y_true, y_pred, num_classes):
    """
    Manually calculates the confusion matrix using NumPy.
    Rows: True Class
    Cols: Predicted Class
    """
    # Ensure inputs are integers
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)

    cm = np.zeros((num_classes, num_classes), dtype=int)

    # Iterate and count (Pure manual implementation)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t][p] += 1

    return cm

def print_classification_report(y_true, y_pred, class_names):
    """
    Manually calculates Precision, Recall, and F1-Score
    to match the look of scikit-learn's classification_report.
    """
    num_classes = len(class_names)
    cm = get_confusion_matrix(y_true, y_pred, num_classes)

    print(f"\n{'':>24} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
    print("-" * 70)

    precisions = []
    recalls = []
    f1s = []
    supports = []

    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        support = np.sum(cm[i, :])

        # Avoid division by zero
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        supports.append(support)

        print(f"{class_names[i]:>24} {precision:10.4f} {recall:10.4f} {f1:10.4f} {support:10d}")

    print("-" * 70)

    # Accuracy
    accuracy = np.sum(np.diag(cm)) / np.sum(cm)
    total_support = np.sum(supports)
    print(f"{'accuracy':>24} {'':>10} {'':>10} {accuracy:10.4f} {total_support:10d}")

    # Weighted Avg
    w_prec = np.sum(np.array(precisions) * np.array(supports)) / total_support
    w_rec = np.sum(np.array(recalls) * np.array(supports)) / total_support
    w_f1 = np.sum(np.array(f1s) * np.array(supports)) / total_support

    print(f"{'weighted avg':>24} {w_prec:10.4f} {w_rec:10.4f} {w_f1:10.4f} {total_support:10d}")

    return accuracy

print("✓ Evaluation Helpers Defined.")

✓ Evaluation Helpers Defined.


In [12]:
# -----------------------------------------------------------
# CELL 7: EXECUTION, COMPARISON & REPORTING
# -----------------------------------------------------------

# 1. Train Models
print("\n" + "="*60)
print("🚀 STARTING TRAINING PHASE")
print("="*60)

# Note: Increase n_estimators for better accuracy if you have time.
# 10 is used here for demonstration speed.
xgb_model = CustomXGBoost(n_estimators=10, learning_rate=0.3, max_depth=3)
xgb_model.fit(X_train, y_train, X_val, y_val)

lgb_model = CustomLightGBM(n_estimators=10, learning_rate=0.3, max_depth=3)
lgb_model.fit(X_train, y_train, X_val, y_val)

cat_model = CustomCatBoost(n_estimators=10, learning_rate=0.3, max_depth=3)
cat_model.fit(X_train, y_train, X_val, y_val)


# 2. Evaluation Function
def evaluate_and_report(models, X_test, y_test, class_names):

    print("\n" + "="*80)
    print("📊 DETAILED PERFORMANCE COMPARISON")
    print("="*80)

    # Header
    print(f"{'Model':<15} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Time (s)':<10}")
    print("-" * 75)

    results = {}

    for name, model in models.items():
        y_pred = model.predict(X_test)

        # Calculate Metrics Manually
        cm = get_confusion_matrix(y_test, y_pred, len(class_names))
        tp = np.diag(cm)
        fp = np.sum(cm, axis=0) - tp
        fn = np.sum(cm, axis=1) - tp

        # Macro Averages
        precision = np.mean(np.divide(tp, tp+fp, out=np.zeros_like(tp, dtype=float), where=(tp+fp)!=0))
        recall = np.mean(np.divide(tp, tp+fn, out=np.zeros_like(tp, dtype=float), where=(tp+fn)!=0))
        f1 = 2 * (precision * recall) / (precision + recall) if (precision+recall)>0 else 0
        acc = np.sum(tp) / np.sum(cm)

        results[name] = {
            'accuracy': acc,
            'preds': y_pred
        }

        print(f"{name:<15} {acc:.4f}     {precision:.4f}     {recall:.4f}     {f1:.4f}     {model.train_time:.2f}")

    # Winner
    winner = max(results, key=lambda x: results[x]['accuracy'])
    print("-" * 75)
    print(f"🏆 WINNER: {winner} ({results[winner]['accuracy']:.2%})")
    print("\n")

    return results, winner

# 3. Run Comparison
models_dict = {
    "Custom XGB": xgb_model,
    "Custom LGBM": lgb_model,
    "Custom CatB": cat_model
}

results, winner_name = evaluate_and_report(models_dict, X_test, y_test, class_names)


# 4. Visualizations
# Bar Chart
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, accs, color=['#4C72B0', '#55A868', '#C44E52'])
plt.ylim(0, 1.05)
plt.title("Model Accuracy Comparison", fontsize=14, fontweight='bold')
plt.ylabel("Accuracy")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2%}", ha='center', fontweight='bold')
plt.show()

# Confusion Matrix of Winner
print(f"\n>>> Detailed Report for {winner_name}:")
y_pred_best = results[winner_name]['preds']
print_classification_report(y_test, y_pred_best, class_names)

# Plot CM
cm = get_confusion_matrix(y_test, y_pred_best, NUM_CLASSES)
plot_advanced_confusion_matrix(cm, class_names, title=f"{winner_name} Confusion Matrix")


🚀 STARTING TRAINING PHASE
Training Custom XGBoost (10 rounds)...
[XGB] |████████████████████████████████████████| 100.0% Acc: 99.83% | Val: 92.64%
Training Custom LightGBM (GOSS)...
[LGBM] |████████████████████████████████████████| 100.0% Acc: 99.94% | Val: 91.03%
Training Custom CatBoost (Ordered)...
[CatB] |████████████████████████████████████████| 100.0% Acc: 98.86% | Val: 91.03%

📊 DETAILED PERFORMANCE COMPARISON
Model           Accuracy   Precision  Recall     F1-Score   Time (s)  
---------------------------------------------------------------------------
Custom XGB      0.9234     0.8164     0.8048     0.8105     641.52
Custom LGBM     0.9234     0.8405     0.8180     0.8291     647.48
Custom CatB     0.9252     0.8677     0.8462     0.8568     758.02
---------------------------------------------------------------------------
🏆 WINNER: Custom CatB (92.52%)





>>> Detailed Report for Custom CatB:

                          precision     recall   f1-score    support
----------------------------------------------------------------------
                       0     0.7647     0.7647     0.7647         17
                       1     0.9641     0.9877     0.9758        326
                       2     0.7778     0.8077     0.7925         52
                       3     0.9091     0.8333     0.8696         24
                       4     0.8611     0.8158     0.8378         38
                       5     0.9294     0.8681     0.8977         91
----------------------------------------------------------------------
                accuracy                           0.9252        548
            weighted avg     0.9249     0.9252     0.9246        548


In [13]:
# -----------------------------------------------------------
# CELL 8: BACKUP SYSTEM (MODELS & RESULTS) - FIXED
# -----------------------------------------------------------
import json
import pickle
import os
from datetime import datetime

print("\n" + "="*60)
print("💾 INITIATING BACKUP PROTOCOL")
print("="*60)

# 1. Create Timestamped Directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_folder = os.path.join(BACKUP_DIR, f"custom_run_{timestamp}")
os.makedirs(backup_folder, exist_ok=True)

print(f"📂 Backup Directory: {backup_folder}")

# 2. Re-Construct Models Dictionary (Fixes NameError)
# We reference the trained model variables directly here
models_to_save = {
    "Custom XGB": xgb_model,
    "Custom LGBM": lgb_model,
    "Custom CatB": cat_model
}

# 3. Save Trained Model Objects
print("\nSaving Model Objects...")
for name, model in models_to_save.items():
    # Create a safe filename (remove spaces)
    filename = f"{name.replace(' ', '_').lower()}.pkl"
    filepath = os.path.join(backup_folder, filename)

    try:
        with open(filepath, "wb") as f:
            pickle.dump(model, f)

        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✅ Saved {name} ({size_kb:.2f} KB)")
    except Exception as e:
        print(f"  ❌ Failed to save {name}: {e}")

# 4. Save Performance Metrics
print("\nSaving Performance Metrics...")
try:
    serializable_results = {}

    # If 'results' variable exists from previous cell, use it
    if 'results' in locals() or 'results' in globals():
        for name, data in results.items():
            serializable_results[name] = {
                'accuracy': float(data['accuracy']),
                # Try to get time, default to 0 if missing
                'training_time': float(getattr(models_to_save.get(name), 'train_time', 0.0))
            }

        # Add Winner Info if available
        if 'winner_name' in locals() or 'winner_name' in globals():
            serializable_results['WINNER'] = winner_name

        results_path = os.path.join(backup_folder, "performance_metrics.json")
        with open(results_path, "w") as f:
            json.dump(serializable_results, f, indent=4)

        print(f"  ✅ Saved metrics to performance_metrics.json")
    else:
        print("  ⚠️ 'results' variable not found. Skipping metrics JSON save.")

except Exception as e:
    print(f"  ⚠️ Could not save metrics JSON: {e}")

print("\n" + "="*60)
print("🎉 BACKUP COMPLETE. ALL DATA SECURED.")
print("="*60)


💾 INITIATING BACKUP PROTOCOL
📂 Backup Directory: /content/drive/MyDrive/FabricDefect/backups/custom_run_20251205_001120

Saving Model Objects...
  ✅ Saved Custom XGB (128.37 KB)
  ✅ Saved Custom LGBM (128.60 KB)
  ✅ Saved Custom CatB (129.92 KB)

Saving Performance Metrics...
  ✅ Saved metrics to performance_metrics.json

🎉 BACKUP COMPLETE. ALL DATA SECURED.
